In [2]:
import os

from dotenv import load_dotenv
load_dotenv()  # loads HF_TOKEN from .env file

In [3]:
! pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

In [4]:
from langchain_community.retrievers import WikipediaRetriever


C:\Users\somil\AppData\Local\Temp\ipykernel_19544\3067170655.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [5]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [6]:

# Define your query
query = "How internet changed the world"

# Get relevant Wikipedia documents

docs = retriever.invoke(query)

In [7]:
docs

[Document(metadata={'title': 'Dead Internet theory', 'summary': 'The dead Internet theory is a concept that asserts that the Internet consists primarily of bot activity and automated content manipulated by algorithmic curation. Originally conceived as a conspiracy theory alleging that the phenomenon is a coordinated effort to control the population and reduce genuine human interaction, the concept is also employed colloquially to describe the impacts of generative AI and emphasize only the core observations without speculating on the driving forces.\nSupporters of the full theory claim that social bots were deliberately created to manipulate algorithms and enhance search results to influence consumers. Some proponents also accuse government agencies of using bots to shape public perception and opinions. The dead Internet theory gained renewed interest following the AI boom that began in the 2020s, with large language model (LLM) chatbots and text-to-image models emerging as technologie

In [8]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
The dead Internet theory is a concept that asserts that the Internet consists primarily of bot activity and automated content manipulated by algorithmic curation. Originally conceived as a conspiracy theory alleging that the phenomenon is a coordinated effort to control the population and reduce genuine human interaction, the concept is also employed colloquially to describe the impacts of generative AI and emphasize only the core observations without speculating on the driving forces.
Supporters of the full theory claim that social bots were deliberately created to manipulate algorithms and enhance search results to influence consumers. Some proponents also accuse government agencies of using bots to shape public perception and opinions. The dead Internet theory gained renewed interest following the AI boom that began in the 2020s, with large language model (LLM) chatbots and text-to-image models emerging as technologies that could theoretically drown out hu

Vector Retriever


In [9]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\somil\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2079.18it/s]


In [10]:
! pip install langchain-chroma chromadb

In [11]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [12]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    collection_name="my_collection"
)

In [13]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [14]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [15]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [16]:
results = vectorstore.similarity_search(query, k=2)

In [17]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


MMR


In [18]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [19]:
from langchain_community.vectorstores import FAISS


# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding
)

In [20]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [21]:
query = "What is langchain?"
results = retriever.invoke(query)

In [22]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
Embeddings are vector representations of text.


Multi-Query Retriever


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
# from langchain.retrievers.multi_query import MultiQueryRetriever

multi query retriever not in new version of langchain  you have to implement on your own
